# Observability for Distributed Systems: metrics, traces and logs that actually find bugs

**DevFest Ilorin 2026** · hands-on companion to the talk by Abdul-Jemeel Odewole · [the repo](https://github.com/intellect4all/devfest-ilorin-2026-observability)

You get a small payment system written in Go: a **transfers** API, a **ledger**, a mock **payout provider** and Postgres.
It has three bugs you can switch on and off. You'll break it and find each bug three ways:

| Signal | Answers | Branch |
|---|---|---|
| Metrics | *Is something wrong?* | `01-metrics` |
| Traces | *Where?* | `02-traces` |
| Logs | *Why?* | `03-logs` |
| Alerts | *Wake me up when users feel it* | `04-alerts` |

Each branch adds one signal to the code. This notebook switches between them in a few seconds.

**How to use it:** run the cells in order with **Shift + Enter**. Read the text between them. The setup cell takes 5 to 7 minutes;
everything after it takes seconds. You don't need Docker or a Google Cloud account, and nothing here costs money.

```
loadgen ──> transfers ──> ledger ──> Postgres
                │
                └──────> provider (mock payout switch)

all services ──OTLP──> Collector ──> Prometheus · Tempo · Loki ──> Grafana
```

## Setup (5 to 7 minutes)

This installs Postgres and the same versions of the OpenTelemetry Collector, Prometheus, Tempo, Loki and Grafana that the talk runs,
builds every branch of the Go code, and loads 3 million ledger entries. It prints each step.

When it finishes, it prints a **Grafana link**. Open it in a new tab and keep it open: no login needed.

In [ ]:
!test -d /content/devfest/colab || (rm -rf /content/devfest && git clone -q -b main https://github.com/intellect4all/devfest-ilorin-2026-observability.git /content/devfest)
import sys, time
sys.path.insert(0, "/content/devfest/colab")
import workshop as w
w.setup()

## Part 0 · The baseline: print debugging

`main` has **no telemetry**. Start steady traffic (20 transfers a second, 90% from one busy *settlement wallet*), send one transfer
yourself, and look at what the service prints. This is what most of us debug with.

In [ ]:
w.load()
time.sleep(10)
w.tx()
w.service_logs("transfers")

Send the **same** transfer twice with the same `Idempotency-Key`. The second call is a replay: same `transfer_id`, no double payout.

In [ ]:
w.tx(key="my-first-key")
w.tx(key="my-first-key")

## Part 1 · Metrics: *is something wrong?*

Switch to `01-metrics`. The code now has OpenTelemetry metrics: request rate, errors and latency for every endpoint (**RED**),
plus goroutines, CPU, memory and the database pool (**USE**).

In Grafana, open **Dashboards → Payments — RED & USE**. Give it a minute to fill.

In [ ]:
w.stage("01-metrics")
w.grafana("d/payments")
time.sleep(45)
w.snapshot()

Healthy: p99 under about 100 ms and no errors. Now break it. **Bug C** is the main incident from the talk.
Turn it on, wait 30 seconds, and look at the dashboard (or the charts below).

In [ ]:
w.bug("c")
time.sleep(30)
w.snapshot()
w.chart(minutes=3)

**What do you see?**

- p99 latency climbs past the 300 ms SLO line and keeps going.
- 5xx stays at 0%. Nothing is *failing*, it's just slow.
- CPU is flat, so the service isn't busy computing.
- **Goroutines climb.** Requests are piling up *inside* the service, waiting for something.

> We know something is wrong. We don't know **where**.

Turn the bug off before moving on.

In [ ]:
w.bug("c", False)

## Part 2 · Traces: *where?*

`02-traces` adds distributed tracing. Every transfer becomes one trace across all three services, with a span for each HTTP call,
each database query and each business step.

In [ ]:
w.stage("02-traces")
time.sleep(20)
w.bug("c")
time.sleep(30)
t = w.trace()

That's the slowest transfer since you turned the bug on. The red bar is the span with the most time *of its own*:
**`account_lock.wait`**. The request spends almost all of its life waiting for a lock before doing any real work.
The ledger and the payout take milliseconds.

**In Grafana:** click the link under the chart to open this same trace in Grafana. Or, on the dashboard's *Latency* panel,
the little diamonds are **exemplars**: real requests. Hover one high on the p99 line and click **View trace**.

> We know **where**. We don't know **why**. Who has the lock?

In [ ]:
w.bug("c", False)

## Part 3 · Logs: *why?*

`03-logs` switches to structured JSON logs with a `trace_id` on every line, shipped to Loki. That one field joins logs to traces.

In [ ]:
w.stage("03-logs")
time.sleep(20)
w.bug("c")
time.sleep(30)
t = w.trace(draw=False)
lines = w.logs(t["traceID"])

Read the `WARN waited for account lock` line:

- `wait_ms`: how long this request waited,
- `held_by_transfer`: **which other transfer** had the lock,
- `held_by_step`: what that transfer was doing. Usually **`payout`**: a network call to the provider.

Somebody is **holding a lock across a network call**. Every payout from the settlement wallet queues behind the previous one.
That's bug C. (Sometimes it says `post_entries`: the holder hadn't reached its payout yet. Run the cell again for another trace.)

**In Grafana:** click the trace link printed above, click the `account_lock.wait` span, then **Related logs**. The same
WARN line appears in Loki, joined to the trace by its `trace_id`.

> Picking a trace yourself from the dashboard? Pick one from **after** you switched to `03-logs` (the right-hand end of the
> graph). The slow traces from Part 2 have no logs: that branch didn't send any.

> Metrics told us something was wrong. Traces told us where. Logs told us why.

In [ ]:
w.bug("c", False)

## Part 4 · Alerts: *wake me up*

`04-alerts` adds two alert rules: **p99 above 300 ms** and **5xx above 5%**. They alert on what users feel, not on causes.
Alerts go to your phone through [ntfy](https://ntfy.sh), a free push service.

1. Install the **ntfy** app (App Store or Google Play).
2. Run the next cell and subscribe to the topic it prints. It's random and only yours.

In [ ]:
w.stage("04-alerts")
w.subscribe()

In [ ]:
w.alert_test()   # your phone should buzz with a test alert

Now the real thing. Turn bug C on and watch the rule go *Pending* → *Firing* (about 30–60 seconds).

In [ ]:
w.bug("c")
w.wait_for_alert()

Turn it off. About 30–60 seconds later the phone gets **Resolved**.

In [ ]:
w.bug("c", False)
time.sleep(45)
w.alerts()

## Part 5 · The hunt: your turn

Switch to `hunt`. The next cell turns on **one of the three bugs at random** and doesn't tell you which.
Use only the signals to find **what** (metrics), **where** (trace) and **why** (logs).

Tools: `w.chart()`, `w.snapshot()`, `w.trace()`, `w.logs(trace_id)`, `w.explain()`, and Grafana.
Stuck? `w.hint()`. Done? `w.reveal()`.

| Bug | Clue in one line |
|---|---|
| A | only hurts under heavy traffic |
| B | only the slowest requests get slower |
| C | you've already met it |

In [ ]:
w.stage("hunt")
w.mystery()
time.sleep(60)   # give the bug a minute to show in the signals

In [ ]:
w.chart(minutes=3)

In [ ]:
t = w.trace()

In [ ]:
lines = w.logs(t["traceID"])

In [ ]:
w.hint()

In [ ]:
w.reveal()

Want to hunt again? Run `w.mystery()` again. To pick a bug yourself: `w.mystery("a")`, `"b"` or `"c"`.

- **Bug A**, an O(n) duplicate check: the `duplicate_check` span is wide with nothing under it; `slow duplicate check`
  with `entries_scanned` in the tens of thousands. It only appears under traffic, so it comes with a surge.
- **Bug B**, a missing index: p99 rises while p50 stays flat; the ledger's `SELECT entries` span is the widest; `w.explain()`
  shows a **Seq Scan**.

## Part 6 · The fixes

`solution` fixes all three bugs in the code. The toggles still answer there, but they change nothing. Turn all three on
plus a traffic surge, and watch p99 stay flat.

In [ ]:
w.stage("solution")
time.sleep(20)
w.bug("a"); w.bug("b"); w.bug("c"); w.surge()
time.sleep(60)
w.snapshot()
w.chart(minutes=3)
w.bugs_off()

How each fix works is in [docs/BUGS.md](https://github.com/intellect4all/devfest-ilorin-2026-observability/blob/main/docs/BUGS.md). The short version:

- **C:** never hold a lock across a network call; release it right after the bookkeeping.
- **B:** the service makes sure the index the query needs exists, every time it starts.
- **A:** a map lookup instead of a scan, with expiry so it can't grow forever.

### See what each signal cost in code

Each branch adds one signal on top of the previous one, so a diff between neighbours is exactly one lesson:

- [main → 01-metrics](https://github.com/intellect4all/devfest-ilorin-2026-observability/compare/main...01-metrics): one `otelhttp` wrapper gives you the whole RED row
- [01-metrics → 02-traces](https://github.com/intellect4all/devfest-ilorin-2026-observability/compare/01-metrics...02-traces)
- [02-traces → 03-logs](https://github.com/intellect4all/devfest-ilorin-2026-observability/compare/02-traces...03-logs)
- [03-logs → 04-alerts](https://github.com/intellect4all/devfest-ilorin-2026-observability/compare/03-logs...04-alerts)
- [04-alerts → solution](https://github.com/intellect4all/devfest-ilorin-2026-observability/compare/04-alerts...solution)

### Run it on your own laptop

With Docker: `git clone https://github.com/intellect4all/devfest-ilorin-2026-observability.git`, then follow the README's quick start (`make up`, `make seed`, `make load`).

### Notes on this notebook

- Colab's free machine has 2 CPUs (the talk's laptop has more), so the database is smaller (3M rows instead of 7M) and the numbers are lower. The shapes are the same.
- If Colab disconnects, run the setup cell again; it skips what's already done.
- `w.status()` shows what's running. `w.service_logs("ledger")` shows a service's output.

In [ ]:
w.status()